# NeuroGolf 7184.44 | Public Blend: Graph-Golf Continuation + task338 hand_sparse

This notebook publishes an audited **7184.44** public artifact for NeuroGolf 2026.

It is a minimal, fully attributed blend of two public lines:

- the **graph-golf continuation** artifact by kokinnwakashuu (which itself adopts the public Ryosuke 7184+ agent bundle plus task101/task118 simplifications);
- the **task338 hand_sparse** model from kunalkarda (dataset neurogolf-cv-v3-track), which replaces the blend task338 at official cost 17939 with an exact model at **16847** (measured gain ln(17939/16847) = +0.0628).

Expected public score: **7184.44** (leaderboard-confirmed by an isolated submission).

Credits: kokinnwakashuu, the Ryosuke 7184+ agent bundle authors, kojimar (blend asset lineage), kunalkarda (task338), yuu111111111 / jcchavez / cocoaAI and the whole graph-golf line for the shared methodology.

## Score model

NeuroGolf charges initializer elements plus intermediate tensor bytes; per task score = max(1, 25 - ln(cost)). For an exact replacement the gain is ln(C_old/C_new).

## Validation

The 400-model archive was checked with: canonical task names and ZIP integrity; full ONNX model checking; the official sanitizer and cost profiler; sanitized ONNX Runtime execution; ALL available examples (train + test + arc-gen) for every changed task; and an isolated leaderboard probe for the single task-level change (task338), so the hidden-set behavior is confirmed, not assumed.

The hidden cell below writes the already validated archive to /kaggle/working/submission.zip. It does not rebuild competition models during execution.

In [ ]:
import base64, hashlib, zipfile
from pathlib import Path
EXPECTED_SHA256 = '6e8ed54f52158be7a11c27451d6608f8faa85f009fe99261ff49118a039d49f6'
EXPECTED_BYTES = 522469
B64 = ''.join([
    '...',
    'BgAAAACQAZABoFoAAC+eBwAAAA==',
])
data = base64.b64decode(B64)
assert len(data) == EXPECTED_BYTES, len(data)
assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256
out = Path('/kaggle/working/submission.zip')
out.write_bytes(data)
zf = zipfile.ZipFile(out)
names = zf.namelist()
assert len(names) == 400 and zf.testzip() is None
print('submission.zip written:', len(data), 'bytes, 400 models, sha ok')
